# 02 — Feature Engineering

This notebook demonstrates the creation and analysis of 5 domain-specific features:
1. **Average bill amount** — mean across 6 months
2. **Credit utilisation ratio** — avg bill / credit limit
3. **Average payment ratio** — avg payment / avg bill
4. **Delay score** — count of delayed months (pay_x > 0)
5. **Payment trend** — slope of payment amounts over time

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_cleaned_data
from src.features import engineer_features
from src.utils import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
# Load cleaned data
df = load_cleaned_data()
print(f'Original shape: {df.shape}')

# Apply feature engineering
df = engineer_features(df)
print(f'After engineering: {df.shape}')

## Engineered Feature Distributions

In [ ]:
eng_cols = ['avg_bill_amt', 'credit_utilization', 'avg_payment_ratio',
            'delay_score', 'payment_trend']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(eng_cols):
    df[col].hist(bins=30, ax=axes[i], edgecolor='black', alpha=0.7, color='#3498db')
    axes[i].set_title(col, fontsize=11)
plt.suptitle('Engineered Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Engineered Features vs. Default

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(eng_cols):
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df['default'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.5, label=f'default={label}',
                     color=color, edgecolor='black')
    axes[i].set_title(col, fontsize=11)
    axes[i].legend(fontsize=8)
plt.suptitle('Engineered Features by Default Status', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Descriptive Statistics

In [ ]:
df[eng_cols].describe().T

## Correlation with Target

In [ ]:
corr_with_target = df[eng_cols + ['default']].corr()['default'].drop('default').sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
corr_with_target.plot(kind='barh', ax=ax, color='#3498db', edgecolor='black')
ax.set_title('Correlation of Engineered Features with Default')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

## Key Takeaways

- **Delay score** has the strongest positive correlation with default.
- **Average payment ratio** is negatively correlated — customers who pay more default less.
- **Credit utilisation** shows a modest positive relationship with default.
- **Payment trend** captures improving / deteriorating payment behaviour over time.